# Dry Bean Variety Classification using Multiclass Neural Network
**Domain:** Agriculture  
**Dataset:** UCI Machine Learning Repository - Dry Bean Dataset (ID: 602)  
**Dataset URL:** [https://archive.ics.uci.edu/dataset/602/dry+bean+dataset](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset)  
**Citation:** Koklu, M. & Ozkan, I. A. (2020). *Multiclass Classification of Dry Beans Using Computer Vision and Machine Learning Techniques.* Computers and Electronics in Agriculture, 174, 105507.  

---
### Academic Project Overview
This project develops and evaluates an Artificial Neural Network (ANN) classifier to distinguish seven commercial dry bean varieties (*Barbunya, Bombay, Cali, Dermason, Horoz, Seker, and Sira*) using 16 morphological and shape features extracted from high-resolution grain imagery.

#### Core Objectives:
1. **Data Ingestion & Integrity Check**: Inspect 13,611 records across 16 geometric features and verify zero missing data.
2. **Strict Leak-Free Preprocessing**: Partition into Stratified 70% Train, 15% Validation, and 15% Test sets. Standardize features by fitting scaler solely on the training partition.
3. **Multiclass Architecture Implementation**: Implement a Multinomial Linear Baseline, a Shallow MLP (1 hidden layer, 64 units), and a Deeper MLP (3 hidden layers: 128 -> 64 -> 32 units with dropout).
4. **Optimization via Stochastic Gradient Descent (SGD)**: Train each network using mini-batch SGD with momentum (mu = 0.9) and cosine annealing learning rate scheduling.
5. **Empirical Evaluation**: Benchmark Test Accuracy, Macro-F1, Weighted-F1, generate loss convergence curves and normalized confusion matrices.


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ensure reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Executing on compute device: {device}')


## 1. Dataset Loading and Exploratory Data Inspection
We fetch the Dry Bean dataset directly from UCI or load from the local cache file `dry_bean.csv`.

In [ ]:
data_path = 'dry_bean.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=602)
    X = dataset.data.features
    y = dataset.data.targets
    df = pd.concat([X, y], axis=1)
    df.to_csv(data_path, index=False)

target_col = [col for col in df.columns if col.lower() in ['class', 'target']][0]
feature_cols = [col for col in df.columns if col != target_col]

print('Dataset Shape:', df.shape)
print(f'Features (N = {len(feature_cols)}):', feature_cols)
print('Missing values:', df.isnull().sum().sum())
df[target_col].value_counts().plot(kind='bar', title='Bean Variety Distribution', color='teal')
plt.ylabel('Count')
plt.show()


## 2. Preprocessing & Leakage-Free Stratified Splitting
To prevent information leakage from test sets into the model, standardization parameters (mean $\mu$ and standard deviation $\sigma$) are computed **only** on the Training set (70%). The Validation (15%) and Test (15%) sets are transformed using these exact training parameters.

In [ ]:
X = df[feature_cols].values
y_raw = df[target_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)

# Stratified Train (70%) and Temp (30%)
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
# Stratified Validation (15%) and Test (15%)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)  # FIT ONLY ON TRAIN
X_val   = scaler.transform(X_val_raw)        # TRANSFORM ONLY
X_test  = scaler.transform(X_test_raw)       # TRANSFORM ONLY

# DataLoaders
batch_size = 64
train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_val,   dtype=torch.float32), torch.tensor(y_val,   dtype=torch.long)), batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test,  dtype=torch.float32), torch.tensor(y_test,  dtype=torch.long)), batch_size=batch_size, shuffle=False)

print(f'Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}')


## 3. Multiclass Neural Network Architectures
We implement three architectures:
1. **Linear Baseline**: Direct projection (Softmax regression) with 119 parameters.
2. **Shallow MLP**: One hidden layer with 64 neurons and ReLU activation (1,543 parameters).
3. **Deeper MLP**: Three hidden layers (128 -> 64 -> 32) with ReLU and 10% Dropout regularization (12,743 parameters).

In [ ]:
class LinearBaseline(nn.Module):
    def __init__(self, in_features=16, num_classes=7):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
    def forward(self, x):
        return self.fc(x)

class ShallowMLP(nn.Module):
    def __init__(self, in_features=16, hidden_dim=64, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class DeeperMLP(nn.Module):
    def __init__(self, in_features=16, num_classes=7, dropout_rate=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)


## 4. Training with Stochastic Gradient Descent (SGD)
SGD updates weights according to:
$$v_{t+1} = \mu v_t + \nabla_\theta \mathcal{L}(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1} - \eta \lambda \theta_t$$
Where $\eta = 0.03$ is the learning rate, $\mu = 0.9$ is momentum, and $\lambda = 10^{-4}$ is weight decay.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=80, lr=0.03, momentum=0.9, weight_decay=1e-4):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.05)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                val_loss += criterion(logits, by).item() * bx.size(0)
                preds = logits.argmax(dim=1)
                correct += (preds == by).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return history


## 5. Model Training Execution
We execute training for 80 epochs for all three models.

In [ ]:
models = {
    'Linear Baseline': LinearBaseline(16, 7),
    'Shallow MLP': ShallowMLP(16, 64, 7),
    'Deeper MLP': DeeperMLP(16, 7, dropout_rate=0.10)
}

histories = {}
for name, mod in models.items():
    print(f'Training {name}...')
    histories[name] = train_model(mod, train_loader, val_loader, epochs=80, lr=0.03)
print('All models trained successfully!')


## 6. Test Set Evaluation & Comparative Metrics

In [ ]:
results = []
test_preds = {}
for name, mod in models.items():
    mod.eval()
    all_preds, all_y = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            bx = bx.to(device)
            preds = mod(bx).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_y.extend(by.numpy())
    test_preds[name] = (all_y, all_preds)
    acc = accuracy_score(all_y, all_preds)
    macro_f1 = f1_score(all_y, all_preds, average='macro')
    weighted_f1 = f1_score(all_y, all_preds, average='weighted')
    results.append({
        'Architecture': name,
        'Test Accuracy (%)': round(acc * 100, 2),
        'Macro-F1': round(macro_f1, 4),
        'Weighted-F1': round(weighted_f1, 4)
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))


## 7. Visualizations: Convergence Curves & Confusion Matrices

In [ ]:
# Convergence Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for name in ['Linear Baseline', 'Shallow MLP', 'Deeper MLP']:
    plt.plot(histories[name]['train_loss'], label=f'{name} (Train)')
plt.title('Training Loss Convergence (SGD)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for name in ['Linear Baseline', 'Shallow MLP', 'Deeper MLP']:
    plt.plot(histories[name]['val_loss'], label=f'{name} (Val)')
plt.title('Validation Loss Convergence (SGD)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (y_true, y_pred)) in enumerate(test_preds.items()):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=False, ax=axes[idx])
    axes[idx].set_title(f'{name}\nNormalized Confusion Matrix')
    axes[idx].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
